In [55]:
import GridMLM_tokenizers
# from GridMLM_tokenizers import get_chord_melody_features
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset

In [56]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [57]:
chord_features = GridMLM_tokenizers.CHORD_FEATURES
chord_id_features = {tokenizer.vocab[k]: v for k, v in chord_features.items()}
print(chord_id_features)

{7: {'quality': 'maj', 'root': 0, 'pitch_classes': [0, 4, 7]}, 36: {'quality': 'maj', 'root': 1, 'pitch_classes': [1, 5, 8]}, 65: {'quality': 'maj', 'root': 2, 'pitch_classes': [2, 6, 9]}, 94: {'quality': 'maj', 'root': 3, 'pitch_classes': [3, 7, 10]}, 123: {'quality': 'maj', 'root': 4, 'pitch_classes': [4, 8, 11]}, 152: {'quality': 'maj', 'root': 5, 'pitch_classes': [0, 5, 9]}, 181: {'quality': 'maj', 'root': 6, 'pitch_classes': [1, 6, 10]}, 210: {'quality': 'maj', 'root': 7, 'pitch_classes': [2, 7, 11]}, 239: {'quality': 'maj', 'root': 8, 'pitch_classes': [0, 3, 8]}, 268: {'quality': 'maj', 'root': 9, 'pitch_classes': [1, 4, 9]}, 297: {'quality': 'maj', 'root': 10, 'pitch_classes': [2, 5, 10]}, 326: {'quality': 'maj', 'root': 11, 'pitch_classes': [3, 6, 11]}, 8: {'quality': 'min', 'root': 0, 'pitch_classes': [0, 3, 7]}, 37: {'quality': 'min', 'root': 1, 'pitch_classes': [1, 4, 8]}, 66: {'quality': 'min', 'root': 2, 'pitch_classes': [2, 5, 9]}, 95: {'quality': 'min', 'root': 3, 'pitch

In [58]:
print(chord_features['D:maj7'])
# cf0 = get_chord_pitch_features(chord_features['D:maj7'])
# print(cf0)

{'quality': 'maj7', 'root': 2, 'pitch_classes': [1, 2, 6, 9]}


In [59]:
train_gjt = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_train'

In [60]:
train_dataset_gjt = CSGridMLMDataset(train_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')

Loading data file.


In [61]:
d = train_dataset_gjt[0]

In [62]:
print(d.keys())
print(d['harmony_ids'])
print(len(d['harmony_ids']))
print(d['pianoroll'].shape)

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity'])
[6, 276, 276, 194, 194, 6, 339, 339, 148, 148, 6, 276, 276, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 159, 159, 6, 332, 332, 129, 129, 6, 274, 274, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 332, 332, 6, 339, 339, 148, 148, 6, 279, 279, 279, 279, 6, 158, 158, 148, 148, 6, 270, 270, 270, 270, 6, 73, 73, 245, 245, 6, 227, 227, 226, 226, 6, 14, 14, 151, 151]
80
(80, 13)


In [63]:
import numpy as np

In [64]:
def bar_split(harmony_ids, pianoroll, tokenizer):
    bars = []
    current_bar = {
        'chord_ids': [],
        'melody_pcs': [],
    }
    for i, hid in enumerate(harmony_ids):
        if hid == tokenizer.vocab['<bar>']:
            if i != 0:  # avoid appending an empty bar at the start
                bars.append(current_bar)
            current_bar = {
                'chord_ids': [],
                'melody_pcs': []
            }
        else:
            current_bar['chord_ids'].append(hid)
            current_bar['melody_pcs'].append(np.where(pianoroll[i] > 0)[0])
    if current_bar:
        bars.append(current_bar)
    return bars

In [65]:
bars = bar_split(d['harmony_ids'], d['pianoroll'], tokenizer)

In [66]:
print(bars)

[{'chord_ids': [276, 276, 194, 194], 'melody_pcs': [array([], dtype=int64), array([ 9, 11]), array([0]), array([0, 4])]}, {'chord_ids': [339, 339, 148, 148], 'melody_pcs': [array([2]), array([ 0, 11]), array([11]), array([11])]}, {'chord_ids': [276, 276, 71, 71], 'melody_pcs': [array([], dtype=int64), array([0, 2]), array([4]), array([4, 9])]}, {'chord_ids': [216, 216, 13, 13], 'melody_pcs': [array([7]), array([4, 5]), array([4]), array([4])]}, {'chord_ids': [159, 159, 159, 159], 'melody_pcs': [array([], dtype=int64), array([5, 7]), array([9]), array([0, 9])]}, {'chord_ids': [332, 332, 129, 129], 'melody_pcs': [array([11]), array([8, 9]), array([8]), array([ 8, 11])]}, {'chord_ids': [274, 274, 71, 71], 'melody_pcs': [array([9]), array([6, 7]), array([6]), array([6, 9])]}, {'chord_ids': [216, 216, 13, 13], 'melody_pcs': [array([7]), array([4, 5]), array([4]), array([4, 7])]}, {'chord_ids': [159, 159, 332, 332], 'melody_pcs': [array([5]), array([3, 4]), array([3]), array([11])]}, {'chord

In [67]:
import torch

class Chord:
    def __init__(self, chord_id, bar_positions, melody_pcs):
        self.chord_id = chord_id
        self.bar_positions = bar_positions
        self.melody_pcs = melody_pcs
        self.get_chord_pitch_features()
        self.get_chord_melody_features()
        # {
        #     'quality': k,
        #     'root': rt,
        #     'pitch_classes': np.where(bin_pcs)[0].tolist()
        # }
    # end init

    def get_chord_pitch_features(self):
        c = chord_id_features[self.chord_id]
        self.pitch_classes = c['pitch_classes']
        self.pcs_map = {}
        self.root = c['root']
        # c is a CHORD_FEATURE dict with keys: 'quality', 'root', 'pitch_classes'
        #
        # returns a tensor of tensors (N, 8), where N is the number of pitches
        # in the chord and 8 is the number of features
        # (root, third, fifth, seventh, extension, chord_pitch (1), melody_pitch (0), offset_from_chord_start (0.0))
        self.graph_features = torch.zeros((len(self.pitch_classes), 8), dtype=torch.float32)
        for i,p in enumerate(self.pitch_classes):
            self.pcs_map[p] = i
            self.graph_features[i] = torch.tensor([
                p == self.root,
                (self.root + 4) % 12 == p or (self.root + 3) % 12 == p,
                (self.root + 7) % 12 == p or (self.root + 6) % 12 == p,
                (self.root + 10) % 12 == p or (self.root + 11) % 12 == p,
                any((self.root + ext) % 12 == p for ext in [1,2,5,8,9]),
                1, 0, 0.0
            ], dtype=torch.float32)
    # end get_chord_pitch_features

    def get_chord_melody_features(self):
        # c is a CHORD_FEATURE dict with keys: 'quality', 'root', 'pitch_classes'
        #
        # returns a tensor of tensors (N, 8), where N is the number of pitches
        # in the melody and 8 is the number of features
        # (root, third, fifth, seventh, extension, chord_pitch (0), melody_pitch (1), offset_from_chord_start (0.0))
        for i, pcs in enumerate(self.melody_pcs):
            for p in pcs:
                if p in self.pcs_map.keys():
                    self.graph_features[self.pcs_map[p], 6] = 1
                else:
                    self.pcs_map[p] = len(self.graph_features)
                    self.pitch_classes.append(p)
                    tmp_feats = torch.tensor([
                        p == self.root,
                        (self.root + 4) % 12 == p or (self.root + 3) % 12 == p,
                        (self.root + 7) % 12 == p or (self.root + 6) % 12 == p,
                        (self.root + 10) % 12 == p or (self.root + 11) % 12 == p,
                        any((self.root + ext) % 12 == p for ext in [1,2,5,8,9]),
                        0, 1, self.bar_positions[i]
                    ], dtype=torch.float32)
                    self.graph_features = torch.cat((self.graph_features, tmp_feats.unsqueeze(0)), dim=0)
    # end get_chord_melody_features

    def print_info(self):
        print(f"Chord label: {tokenizer.ids_to_tokens[self.chord_id]}")
        print(f"Pitch classes: {self.pitch_classes}")
        print(f"Root: {self.root}")
        print(f"Chord ID: {self.chord_id}")
        print(f"Bar Positions: {self.bar_positions}")
        print(f"Melody PCs: {self.melody_pcs}")
        print(f"Graph Features:\n{self.graph_features}")
    # end print_info
# end class Chord

In [68]:
def make_chord_objects(bars):
    bars_out = []
    for bar in bars:
        chord_objects = []
        chord_ids = bar['chord_ids']
        melody_pcs = bar['melody_pcs']
        tmp_positions = []
        tmp_melody_pcs = []
        if len(chord_ids) > 0:
            for i in range(len(chord_ids)):
                hid = chord_ids[i]
                if i > 0:
                    if hid == chord_ids[i-1]:
                        tmp_positions.append(i)
                        tmp_melody_pcs.append(melody_pcs[i])
                    else:
                        chord_objects.append(Chord(chord_ids[i-1], tmp_positions, tmp_melody_pcs))
                        tmp_positions = [i]
                        tmp_melody_pcs = [melody_pcs[i]]
                else:
                    tmp_positions.append(i)
                    tmp_melody_pcs.append(melody_pcs[i])
            # end for
            chord_objects.append(Chord(hid, tmp_positions, tmp_melody_pcs))
        bars_out.append(chord_objects)
    return bars_out

In [69]:
bars_chord_objects = make_chord_objects(bars)

In [71]:
bar_idx = 3
print(bars[bar_idx])
for i, c in enumerate(bars_chord_objects[bar_idx]):
    print(f"Chord {i}:")
    bars_chord_objects[bar_idx][i].print_info()

{'chord_ids': [216, 216, 13, 13], 'melody_pcs': [array([7]), array([4, 5]), array([4]), array([4])]}
Chord 0:
Chord label: G:7
Pitch classes: [2, 5, 7, 11, 4]
Root: 7
Chord ID: 216
Bar Positions: [0, 1]
Melody PCs: [array([7]), array([4, 5])]
Graph Features:
tensor([[0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 1., 0.],
        [1., 0., 0., 0., 0., 1., 1., 0.],
        [0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 1., 1., 0.]])
Chord 1:
Chord label: C:7
Pitch classes: [0, 4, 7, 10]
Root: 0
Chord ID: 13
Bar Positions: [2, 3]
Melody PCs: [array([4]), array([4])]
Graph Features:
tensor([[1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 1., 1., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.]])
